# AI Programming — Lecture 19
## Lab 4-4b: Decoder-only Transformer for ETTh1 Forecasting
### Residual Prediction

앞의 Direct Prediction과 동일한 decoder-only Transformer를 사용하되,
미래값 자체가 아니라 **마지막 관측값 기준 residual**을 예측합니다.

$$
r_{t+k} = x_{t+k} - x_t
$$

최종 예측은

$$
\hat{x}_{t+k} = x_t + \hat{r}_{t+k}
$$

입니다.

### 학습 목표
- Direct prediction과 residual prediction의 차이를 이해합니다.
- Last-value anchor를 기준으로 residual target을 구성할 수 있습니다.
- Autoregressive residual forecasting을 수행할 수 있습니다.
- Direct / Residual / Last-Value baseline을 비교할 수 있습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

drive.mount('/content/drive')

## 1. ETTh1 데이터 불러오기

In [ ]:
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv'

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.head())
print(df.isna().sum())

ot = df[['OT']].values.astype('float32')


## 2. Chronological Split과 Standardization

In [ ]:
CONTEXT_LEN = 96
PRED_LEN = 24
MODEL_LEN = CONTEXT_LEN + PRED_LEN - 1   # 119

n = len(ot)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_raw = ot[:train_end]
val_raw = ot[train_end:val_end]
test_raw = ot[val_end:]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

print('Train:', train_raw.shape)
print('Validation:', val_raw.shape)
print('Test:', test_raw.shape)

## 3. Teacher-Forcing Sequence 구성

In [ ]:
def create_decoder_sequences(values, context_len=96, pred_len=24):
    total_len = context_len + pred_len
    X, Y = [], []

    for i in range(len(values) - total_len + 1):
        window = values[i:i + total_len]

        # Past 96 + first 23 future values
        X.append(window[:-1])

        # Next 24 absolute future values
        Y.append(window[context_len:])

    return (
        np.array(X, dtype='float32'),
        np.array(Y, dtype='float32')
    )

X_train, y_train = create_decoder_sequences(train_scaled, CONTEXT_LEN, PRED_LEN)
X_val, y_val = create_decoder_sequences(val_scaled, CONTEXT_LEN, PRED_LEN)
X_test, y_test = create_decoder_sequences(test_scaled, CONTEXT_LEN, PRED_LEN)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_val  :', X_val.shape)
print('y_val  :', y_val.shape)
print('X_test :', X_test.shape)
print('y_test :', y_test.shape)

## 4. Residual Prediction Target

각 sample의 마지막 관측값을 baseline으로 두고,
미래값에서 baseline을 뺀 residual을 학습 target으로 사용합니다.

In [ ]:
train_base = X_train[:, CONTEXT_LEN - 1:CONTEXT_LEN, :]
val_base = X_val[:, CONTEXT_LEN - 1:CONTEXT_LEN, :]
test_base = X_test[:, CONTEXT_LEN - 1:CONTEXT_LEN, :]

y_train_res = y_train - train_base
y_val_res = y_val - val_base
y_test_res = y_test - test_base

print('Residual target shape:', y_train_res.shape)
print('Residual mean:', y_train_res.mean())
print('Residual std :', y_train_res.std())

## 5. Learned Positional Embedding

In [ ]:
class LearnedPositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, inputs):
        positions = keras.ops.arange(
            0, keras.ops.shape(inputs)[1], 1
        )
        return inputs + self.position_embedding(positions)

## 6. Decoder-only Transformer Block

In [ ]:
class DecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(ff_dim, activation='relu')
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs):
        attention_output = self.attention(
            inputs,
            inputs,
            use_causal_mask=True
        )

        x = self.norm1(
            inputs + self.dropout1(attention_output)
        )

        ffn_output = self.dense2(
            self.dense1(x)
        )

        return self.norm2(
            x + self.dropout2(ffn_output)
        )

## 7. Residual Decoder-only Model 구성

In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

inputs = keras.Input(shape=(MODEL_LEN, 1))

projection_layer = layers.Dense(EMBED_DIM)
x = projection_layer(inputs)

position_layer = LearnedPositionalEmbedding(
    MODEL_LEN, EMBED_DIM
)
x = position_layer(x)

decoder1 = DecoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
x = decoder1(x)

decoder2 = DecoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
x = decoder2(x)

residual_layer = layers.Dense(1)
all_residuals = residual_layer(x)

backbone = keras.Model(
    inputs,
    all_residuals,
    name='residual_decoder_backbone'
)

forecast_layer = layers.Lambda(
    lambda x: x[:, -PRED_LEN:, :]
)
forecast_residuals = forecast_layer(all_residuals)

model = keras.Model(
    inputs,
    forecast_residuals,
    name='residual_decoder_forecaster'
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

model.summary()

## 8. Model Training

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train_res,
    validation_data=(X_val, y_val_res),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Residual MSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()

## 9. Batched Autoregressive Residual Inference

각 step에서 residual을 예측한 뒤,
마지막 관측값을 다시 더해 실제 scale의 다음 값을 만듭니다.

In [ ]:
def autoregressive_residual_forecast_batch(
    backbone,
    past_batch,
    context_len=96,
    pred_len=24
):
    batch_size = past_batch.shape[0]

    buffer = np.zeros(
        (batch_size, context_len + pred_len - 1, 1),
        dtype='float32'
    )

    buffer[:, :context_len, :] = past_batch

    # Fixed baseline for the whole forecast horizon
    baseline = past_batch[:, -1, 0]

    predictions = np.zeros(
        (batch_size, pred_len),
        dtype='float32'
    )

    for step in range(pred_len):
        outputs = backbone(buffer, training=False).numpy()

        output_position = context_len - 1 + step
        residual = outputs[:, output_position, 0]

        next_value = baseline + residual
        predictions[:, step] = next_value

        if step < pred_len - 1:
            input_position = context_len + step
            buffer[:, input_position, 0] = next_value

    return predictions

## 10. Test Set 평가

In [ ]:
EVAL_BATCH_SIZE = 256

pred_batches = []

for start in range(0, len(X_test), EVAL_BATCH_SIZE):
    end = min(start + EVAL_BATCH_SIZE, len(X_test))

    past_batch = X_test[
        start:end,
        :CONTEXT_LEN,
        :
    ]

    pred_batch = autoregressive_residual_forecast_batch(
        backbone,
        past_batch,
        CONTEXT_LEN,
        PRED_LEN
    )

    pred_batches.append(pred_batch)

y_pred_ar = np.concatenate(pred_batches, axis=0)

y_test_2d = y_test[..., 0]

norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    y_pred_ar.reshape(-1)
)
norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    y_pred_ar.reshape(-1)
)

y_test_real = scaler.inverse_transform(
    y_test_2d.reshape(-1, 1)
).reshape(y_test_2d.shape)

y_pred_real = scaler.inverse_transform(
    y_pred_ar.reshape(-1, 1)
).reshape(y_pred_ar.shape)

real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)
real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)

print(f'Normalized MSE : {norm_mse:.4f}')
print(f'Normalized MAE : {norm_mae:.4f}')
print(f'MSE (°C²)      : {real_mse:.4f}')
print(f'MAE (°C)       : {real_mae:.4f}')

## 11. Last-Value Baseline

In [ ]:
last_value_pred = np.repeat(
    X_test[:, CONTEXT_LEN - 1, 0][:, None],
    PRED_LEN,
    axis=1
)

baseline_norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)
baseline_norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)

last_value_real = scaler.inverse_transform(
    last_value_pred.reshape(-1, 1)
).reshape(last_value_pred.shape)

baseline_real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)
baseline_real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)

print('Last-Value Baseline')
print(f'Normalized MSE : {baseline_norm_mse:.4f}')
print(f'Normalized MAE : {baseline_norm_mae:.4f}')
print(f'MSE (°C²)      : {baseline_real_mse:.4f}')
print(f'MAE (°C)       : {baseline_real_mae:.4f}')

## 12. Direct Prediction과 비교

같은 Transformer 구조에서 prediction target만 바꾸었을 때
forecast drift와 error가 어떻게 달라지는지 확인합니다.

In [ ]:
previous_direct_mse = 14.2468
previous_direct_mae = 3.1329

print('Previous direct prediction')
print(f'MSE (°C²): {previous_direct_mse:.4f}')
print(f'MAE (°C) : {previous_direct_mae:.4f}')

print()
print('Residual + learned PE')
print(f'MSE (°C²): {real_mse:.4f}')
print(f'MAE (°C) : {real_mae:.4f}')

print()
print('Last-value baseline')
print(f'MSE (°C²): {baseline_real_mse:.4f}')
print(f'MAE (°C) : {baseline_real_mae:.4f}')

## 13. Forecast Example

In [ ]:
sample_idx = 0

past_real = scaler.inverse_transform(
    X_test[sample_idx, :CONTEXT_LEN]
).reshape(-1)

future_real = y_test_real[sample_idx]
pred_real = y_pred_real[sample_idx]

past_x = np.arange(-CONTEXT_LEN + 1, 1)
future_x = np.arange(1, PRED_LEN + 1)

plt.figure(figsize=(10, 4))
plt.plot(past_x, past_real, label='Past OT')
plt.plot(future_x, future_real, label='Ground Truth')
plt.plot(future_x, pred_real, label='Residual Forecast')

plt.axvline(0, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel('OT (°C)')
plt.title('Autoregressive Residual Forecast')
plt.legend()
plt.grid(True)
plt.show()

## 핵심 정리

- Direct: 미래값 자체를 예측
- Residual: 마지막 관측값에서의 변화량을 예측
- Residual formulation은 autoregressive drift를 줄이는 데 도움이 될 수 있습니다.
- 반드시 last-value baseline과 함께 비교해야 합니다.